# 52 — Chọn, lọc, và Copy-on-Write

Notebook này gồm hai nửa. Nửa đầu là kỹ năng nền: `[]`, `.loc`, `.iloc`,
boolean mask, `query()`. Nửa sau là **thay đổi lớn nhất của pandas 3.0**:

> **Copy-on-Write luôn bật và không tắt được nữa.**

Hệ quả: một số dòng code từng chạy đúng ở pandas 2.x giờ **không làm gì cả**.
Không exception, không lỗi — chỉ là giá trị không được ghi. Notebook đo cụ thể
ba tình huống, và một trong ba **không có cảnh báo nào**.

In [1]:
import sys
import warnings
from pathlib import Path

GOC = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "finlens_examples").is_dir())
sys.path.insert(0, str(GOC))

import numpy as np
import pandas as pd

import finlens
from finlens_examples import hom_nay, lui_ngay

client = finlens.client()
HOM_NAY = hom_nay(client)

gia = client.eod.stock.ohlcv(["HPG", "VCB", "FPT"], start=lui_ngay(HOM_NAY, thang=6))
print(f"pandas {pd.__version__} · frame {gia.shape[0]} dòng × {gia.shape[1]} cột")

pandas 3.0.5 · frame 363 dòng × 7 cột


## 1 · Ba cách chọn, và cách chọn sai

| Cú pháp | Chọn theo | Trả về |
|---|---|---|
| `df["close"]` | tên cột | `Series` |
| `df[["close", "volume"]]` | tên cột | `DataFrame` |
| `df.loc[hàng, cột]` | **nhãn** | tuỳ đầu vào |
| `df.iloc[hàng, cột]` | **vị trí** | tuỳ đầu vào |
| `df[mask]` | boolean | `DataFrame` |

In [2]:
print(f"df['close']            → {type(gia['close']).__name__}")
print(f"df[['close']]          → {type(gia[['close']]).__name__}")
print(f"df.loc[0, 'close']     → {type(gia.loc[0, 'close']).__name__}  = {gia.loc[0, 'close']}")
print(f"df.loc[0:2, 'close']   → {type(gia.loc[0:2, 'close']).__name__}, {len(gia.loc[0:2, 'close'])} phần tử")
print(f"df.iloc[0:2]['close']  → {len(gia.iloc[0:2])} phần tử")

df['close']            → Series
df[['close']]          → DataFrame
df.loc[0, 'close']     → float64  = 94.99
df.loc[0:2, 'close']   → Series, 3 phần tử
df.iloc[0:2]['close']  → 2 phần tử


⚠️ Để ý hai dòng cuối: `loc[0:2]` cho **3** phần tử còn `iloc[0:2]` cho **2**.
`.loc` bao gồm cả điểm cuối vì nó cắt theo **nhãn**, và nhãn cuối là một nhãn
thật bạn vừa nêu tên. `.iloc` theo quy ước slice của Python, không bao gồm.

Đây là khác biệt duy nhất trong pandas mà tôi khuyên học thuộc — nó lệch đúng
một dòng, và một dòng thì không ai nhìn thấy.

In [3]:
print(f"loc[0:2] → nhãn {gia.loc[0:2].index.tolist()}   ({len(gia.loc[0:2])} dòng, BAO GỒM 2)")
print(f"iloc[0:2] → nhãn {gia.iloc[0:2].index.tolist()}      ({len(gia.iloc[0:2])} dòng, KHÔNG bao gồm 2)")

loc[0:2] → nhãn [0, 1, 2]   (3 dòng, BAO GỒM 2)
iloc[0:2] → nhãn [0, 1]      (2 dòng, KHÔNG bao gồm 2)


## 2 · Boolean mask — cách lọc chính

Mask là một `Series` boolean cùng độ dài. Toán tử là `&`, `|`, `~` — **không
phải** `and`, `or`, `not`.

In [4]:
mask = (gia["symbol"] == "HPG") & (gia["close"] > 23)

print(f"mask: {type(mask).__name__} dtype={mask.dtype}, {mask.sum()}/{len(mask)} dòng True")
print(f"Lọc ra: {len(gia[mask])} dòng")

try:
    gia[(gia["symbol"] == "HPG") and (gia["close"] > 23)]
except Exception as e:
    print(f"\nDùng `and` thay `&` → {type(e).__name__}: {str(e)[:90]}")

mask: Series dtype=boolean, 95/363 dòng True
Lọc ra: 95 dòng

Dùng `and` thay `&` → ValueError: The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.al


`and` gọi `bool()` lên cả `Series`, mà một Series nhiều phần tử thì không có
giá trị chân lý duy nhất — nên pandas ném lỗi thay vì đoán. Đây là một trong ít
chỗ pandas chọn ném lỗi to thay vì im lặng.

⚠️ Và **luôn bọc ngoặc từng điều kiện**: `&` có độ ưu tiên cao hơn `>`.

In [5]:
try:
    gia["symbol"] == "HPG" & gia["close"] > 23
except Exception as e:
    print(f"Thiếu ngoặc → {type(e).__name__}: {str(e)[:80]}")

Thiếu ngoặc → TypeError: Cannot perform 'rand_' with a dtyped [float64] array and scalar of type [bool]


### `query()` — dễ đọc hơn khi nhiều điều kiện

In [6]:
a = gia[(gia["symbol"] == "HPG") & (gia["close"] > 23) & (gia["volume"] > 20e6)]
b = gia.query("symbol == 'HPG' and close > 23 and volume > 20e6")

print(f"mask thường: {len(a)} dòng")
print(f"query()    : {len(b)} dòng   ← cùng kết quả: {a.equals(b)}")
print()
print("query() dùng biến ngoài bằng tiền tố @:")
NGUONG = 23
print(f"  {len(gia.query('symbol == \"HPG\" and close > @NGUONG'))} dòng")

mask thường: 67 dòng
query()    : 67 dòng   ← cùng kết quả: True

query() dùng biến ngoài bằng tiền tố @:
  95 dòng


## 3 · ⚠️ Copy-on-Write — nửa sau của notebook

Từ pandas 3.0, Copy-on-Write **luôn bật** và cái công tắc tắt nó đã bị bỏ.

In [7]:
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter("always")
    pd.options.mode.copy_on_write = False  # thử tắt đi xem sao
    if w:
        print(f"{w[0].category.__name__}:")
        print(f"  {w[0].message}")

print()
print(f"Đọc lại option: {pd.options.mode.copy_on_write}   ← nhận giá trị, nhưng vô hiệu")

# Chứng minh CoW vẫn bật bất chấp option
thu_tat = pd.DataFrame({"a": [1, 2, 3]})
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter("always")
    thu_tat["a"][0] = 999
print(f"Sau khi 'tắt' CoW, chained assignment: {thu_tat['a'].tolist()}")
print(f"  → vẫn KHÔNG ghi, và vẫn cảnh báo {[x.category.__name__ for x in w]}")

pd.options.mode.copy_on_write = True  # trả lại, tránh để rác cho cell sau

  The 'mode.copy_on_write' option is deprecated. Copy-on-Write can no longer be disabled (it is always enabled with pandas >= 3.0), and setting the option has no impact. This option will be removed in pandas 4.0.

Đọc lại option: False   ← nhận giá trị, nhưng vô hiệu
Sau khi 'tắt' CoW, chained assignment: [1, 2, 3]
  → vẫn KHÔNG ghi, và vẫn cảnh báo ['ChainedAssignmentError']


C:\Users\hung.hm\AppData\Local\Temp\ipykernel_12560\2311112695.py:9: Pandas4Warning: The 'mode.copy_on_write' option is deprecated. Copy-on-Write can no longer be disabled (it is always enabled with pandas >= 3.0), and setting the option has no impact. This option will be removed in pandas 4.0.
  print(f"Đọc lại option: {pd.options.mode.copy_on_write}   ← nhận giá trị, nhưng vô hiệu")
C:\Users\hung.hm\AppData\Local\Temp\ipykernel_12560\2311112695.py:19: Pandas4Warning: The 'mode.copy_on_write' option is deprecated. Copy-on-Write can no longer be disabled (it is always enabled with pandas >= 3.0), and setting the option has no impact. This option will be removed in pandas 4.0.
  pd.options.mode.copy_on_write = True  # trả lại, tránh để rác cho cell sau


Option vẫn **nhận** giá trị `False` — nó chưa bị xoá khỏi API — nhưng nó không
còn tác dụng gì. Đây là dạng bẫy riêng của nó: code cũ có dòng
`pd.options.mode.copy_on_write = False` để giữ hành vi pandas 2.x sẽ chạy trơn
tru, không lỗi, và **không đạt được điều nó định làm**.

### Quy tắc CoW gói trong một câu

> **Mọi thứ bạn lấy ra từ một DataFrame đều hành xử như một bản sao.**
> Ghi vào nó không bao giờ chạm tới frame gốc.

Điều đó nghe hiển nhiên và an toàn. Vấn đề là code cũ **dựa vào điều ngược
lại**, và pandas không thể vừa giữ tương thích vừa sửa được.

### Tình huống 1 — chained assignment: có cảnh báo, không ghi

Đây là dạng phổ biến nhất trong code pandas 2.x.

In [8]:
df = gia.groupby("symbol", observed=True).head(2).copy()
truoc = df["close"].tolist()

with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter("always")
    df["close"][0] = 999.0  # ← cú pháp kinh điển của pandas 2.x
    canh_bao = [x.category.__name__ for x in w]

print(f"Trước: {truoc}")
print(f"Sau  : {df['close'].tolist()}")
print(f"\nCảnh báo: {canh_bao}")
print(f"Giá trị 999 có được ghi không? {999.0 in df['close'].tolist()}")

Trước: [94.99, 93.22, 23.97, 24.55, 63.87, 65.85]
Sau  : [94.99, 93.22, 23.97, 24.55, 63.87, 65.85]

Cảnh báo: ['ChainedAssignmentError']
Giá trị 999 có được ghi không? False


`df["close"][0] = 999` chạy hai bước: `df["close"]` tạo ra một Series **mới**,
rồi `[0] = 999` ghi vào Series mới đó. Series ấy bị vứt đi ngay sau dòng lệnh.
Frame gốc chưa từng được chạm tới.

Ở pandas 2.x đôi khi nó *có* ghi được — tuỳ vào việc pandas trả về view hay
copy, mà điều đó lại tuỳ vào bố cục bộ nhớ. Chính sự "đôi khi" đó là lý do
CoW ra đời.

**Cách đúng: một lệnh `.loc` duy nhất.**

In [9]:
df.loc[df.index[0], "close"] = 999.0
print(f"Dùng .loc → {df['close'].tolist()}")
print(f"999 đã được ghi: {999.0 in df['close'].tolist()}")

Dùng .loc → [999.0, 93.22, 23.97, 24.55, 63.87, 65.85]
999 đã được ghi: True


### Tình huống 2 — ghi vào kết quả lọc: **không có cảnh báo nào**

Đây mới là tình huống nguy hiểm, vì nó im lặng hoàn toàn.

In [10]:
# ⚠️ gia.head(20) toàn FPT vì frame sắp theo mã — lấy vài dòng của MỖI mã
df2 = gia.groupby("symbol", observed=True).head(4).copy()
loc_ra = df2[df2["symbol"] == "HPG"]

with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter("always")
    loc_ra["close"] = 0.0
    so_canh_bao = len(w)

print(f"Số cảnh báo: {so_canh_bao}")
print(f"loc_ra đã đổi:   {loc_ra['close'].unique().tolist()}")
print(f"df2 gốc có đổi?  {sorted(df2[df2['symbol'] == 'HPG']['close'].unique().tolist())[:3]} …")
print(f"\n→ Ghi thành công vào bản lọc, nhưng frame GỐC không đổi. Và không ai báo bạn.")

Số cảnh báo: 0
loc_ra đã đổi:   [0.0]
df2 gốc có đổi?  [23.97, 24.55, 25.27] …

→ Ghi thành công vào bản lọc, nhưng frame GỐC không đổi. Và không ai báo bạn.


Người viết code thường muốn sửa **frame gốc**. Trước pandas 3.0 đây là chỗ sinh
ra `SettingWithCopyWarning` — cảnh báo nổi tiếng vì hay báo nhầm nên ai cũng tắt
nó đi. CoW làm hành vi trở nên **nhất quán** (luôn là bản sao), nên pandas không
còn cần cảnh báo nữa.

Nhất quán thì dễ suy luận hơn. Nhưng code cũ dựa vào hành vi kia sẽ hỏng lặng lẽ.

**Cách đúng: `.loc` với mask, một lệnh.**

In [11]:
df3 = gia.groupby("symbol", observed=True).head(4).copy()
df3.loc[df3["symbol"] == "HPG", "close"] = 0.0

print(f"df3 sau .loc[mask, cột]: HPG có close = {df3[df3['symbol'] == 'HPG']['close'].unique().tolist()}")
print(f"các mã khác giữ nguyên: {df3[df3['symbol'] != 'HPG']['close'].head(3).tolist()}")

df3 sau .loc[mask, cột]: HPG có close = [0.0]
các mã khác giữ nguyên: [94.99, 93.22, 91.24]


### Tình huống 3 — hàm nhận frame rồi sửa nó

Dạng này hay xuất hiện trong pipeline nhiều bước.

In [12]:
def them_cot_sai(d: pd.DataFrame) -> None:
    """Sửa tại chỗ — hành vi này KHÔNG còn đáng tin ở pandas 3.0."""
    d["gtgd"] = d["close"] * d["volume"] * 1_000


def them_cot_dung(d: pd.DataFrame) -> pd.DataFrame:
    """Trả về frame mới — cách duy nhất còn đúng chắc chắn."""
    return d.assign(gtgd=d["close"] * d["volume"] * 1_000)


goc = gia.groupby("symbol", observed=True).head(4).copy()

lat_cat = goc[goc["symbol"] == "HPG"]  # một lát cắt, không phải bản sao rõ ràng
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter("always")
    them_cot_sai(lat_cat)
    print(f"Sửa tại chỗ trên lát cắt: {len(w)} cảnh báo")

print(f"  lát cắt có cột gtgd? {'gtgd' in lat_cat.columns}")
print(f"  frame gốc có?        {'gtgd' in goc.columns}   ← người gọi hàm sẽ ngạc nhiên")

moi = them_cot_dung(goc)
print(f"\nDùng assign(): frame trả về có gtgd? {'gtgd' in moi.columns}, gốc có? {'gtgd' in goc.columns}")

Sửa tại chỗ trên lát cắt: 0 cảnh báo
  lát cắt có cột gtgd? True
  frame gốc có?        False   ← người gọi hàm sẽ ngạc nhiên

Dùng assign(): frame trả về có gtgd? True, gốc có? False


**Quy tắc thực hành:** hàm biến đổi dữ liệu nên **nhận vào và trả ra**, đừng
sửa tại chỗ. Nó đúng ở mọi phiên bản pandas, dễ test hơn, và nối chuỗi được.

## 4 · CoW không làm chậm — nó làm nhanh hơn

Cái tên gợi ý "sao chép", nên nhiều người tưởng nó tốn kém. Ngược lại: pandas
chỉ **thật sự sao chép khi có ai đó ghi**, còn đọc thì dùng chung bộ nhớ.

In [13]:
import time

lon = client.eod.stock.ohlcv(
    client.meta.symbols(exchange="HOSE", kind="stock")["symbol"].tolist()[:150],
    start=lui_ngay(HOM_NAY, nam=1),
)
print(f"Frame: {len(lon):,} dòng")

t0 = time.perf_counter()
for _ in range(50):
    _ = lon[lon["close"] > 20]
t_loc = time.perf_counter() - t0

t0 = time.perf_counter()
for _ in range(50):
    _ = lon.copy()
t_copy = time.perf_counter() - t0

print(f"50 lần lọc (chỉ đọc)  : {t_loc * 1000:>7.0f} ms")
print(f"50 lần .copy() tường minh: {t_copy * 1000:>7.0f} ms")
print(f"→ lọc rẻ hơn {t_copy / t_loc:.1f} lần vì CoW chưa cần sao chép gì")

Frame: 36,723 dòng
50 lần lọc (chỉ đọc)  :      62 ms
50 lần .copy() tường minh:      97 ms
→ lọc rẻ hơn 1.6 lần vì CoW chưa cần sao chép gì


C:\Program Files\Python312\Lib\asyncio\base_events.py:1999: PartialDataWarning: 1 mã không lấy được: DMX. Chi tiết ở `df.attrs["finlens"]["failed"]`. Dùng `on_error="raise"` để biến thành ngoại lệ.
  handle._run()


In [14]:
sub = lon["close"]
print(f"Series lấy ra có dùng chung bộ nhớ với frame không? {np.shares_memory(sub.to_numpy(), lon['close'].to_numpy())}")
print("→ chia sẻ, cho tới khi có ai ghi vào một trong hai")

Series lấy ra có dùng chung bộ nhớ với frame không? True
→ chia sẻ, cho tới khi có ai ghi vào một trong hai


## 5 · `.copy()` — khi nào vẫn cần

CoW làm `.copy()` bớt cần thiết, nhưng chưa thừa. Vẫn gọi nó khi bạn định
**ghi nhiều lần** vào một lát cắt và muốn nói rõ ý định:

In [15]:
# Cách rõ ràng: tách hẳn ra rồi sửa thoải mái
rieng = gia[gia["symbol"] == "HPG"].copy()
rieng["ls"] = rieng["close"].pct_change() * 100
rieng["ma5"] = rieng["close"].rolling(5).mean()

print(f"Đã thêm {['ls', 'ma5']} vào bản riêng — {len(rieng)} dòng")
print(f"Frame gốc vẫn {gia.shape[1]} cột")

Đã thêm ['ls', 'ma5'] vào bản riêng — 121 dòng
Frame gốc vẫn 7 cột


⚠️ Nếu bỏ `.copy()` ở dòng đầu, hai dòng sau **vẫn chạy** và vẫn cho kết quả
đúng trên `rieng` — nhưng pandas phải sao chép ngầm ở lần ghi đầu tiên, và bạn
mất đi cơ hội nói rõ ý định cho người đọc code. `.copy()` ở đây là **tài liệu**
nhiều hơn là kỹ thuật.

## 6 · Bảng chuyển đổi từ pandas 2.x

Sáu mẫu hay gặp nhất, và cách viết lại.

In [16]:
bang = pd.DataFrame(
    [
        ("df['a'][0] = x", "df.loc[df.index[0], 'a'] = x", "có cảnh báo, không ghi"),
        ("df[mask]['a'] = x", "df.loc[mask, 'a'] = x", "KHÔNG cảnh báo, không ghi vào gốc"),
        ("df.a[0] = x", "df.loc[df.index[0], 'a'] = x", "có cảnh báo, không ghi"),
        ("sub = df[m]; sub['a'] = x", "sub = df[m].copy(); sub['a'] = x", "ghi vào sub, gốc không đổi"),
        ("def f(d): d['a'] = x", "def f(d): return d.assign(a=x)", "tuỳ ngữ cảnh, không đáng tin"),
        ("df.iloc[0]['a'] = x", "df.iloc[0, df.columns.get_loc('a')] = x", "không ghi"),
    ],
    columns=["pandas 2.x", "pandas 3.0", "chuyện gì xảy ra nếu giữ nguyên"],
)
bang

,pandas 2.x,pandas 3.0,chuyện gì xảy ra nếu giữ nguyên
0,df['a'][0] = x,"df.loc[df.index[0], 'a'] = x","có cảnh báo, không ghi"
1,df[mask]['a'] = x,"df.loc[mask, 'a'] = x","KHÔNG cảnh báo, không ghi vào gốc"
2,df.a[0] = x,"df.loc[df.index[0], 'a'] = x","có cảnh báo, không ghi"
3,sub = df[m]; sub['a'] = x,sub = df[m].copy(); sub['a'] = x,"ghi vào sub, gốc không đổi"
4,def f(d): d['a'] = x,def f(d): return d.assign(a=x),"tuỳ ngữ cảnh, không đáng tin"
5,df.iloc[0]['a'] = x,"df.iloc[0, df.columns.get_loc('a')] = x",không ghi


## 7 · Bắt lỗi này trong code có sẵn

Đừng đọc tay từng dòng. Biến cảnh báo thành lỗi rồi chạy test là xong.

In [17]:
print("Trong pytest, thêm vào pyproject.toml hoặc pytest.ini:")
print("""
    [tool.pytest.ini_options]
    filterwarnings = [
        "error::pandas.errors.ChainedAssignmentError",
    ]
""")

print("Hoặc ngay trong Python:")
print('    warnings.simplefilter("error", pd.errors.ChainedAssignmentError)')

Trong pytest, thêm vào pyproject.toml hoặc pytest.ini:

    [tool.pytest.ini_options]
    filterwarnings = [
        "error::pandas.errors.ChainedAssignmentError",
    ]

Hoặc ngay trong Python:
    warnings.simplefilter("error", pd.errors.ChainedAssignmentError)


In [18]:
# Thử ngay tại đây
with warnings.catch_warnings():
    warnings.simplefilter("error", pd.errors.ChainedAssignmentError)
    thu = gia.head(3).copy()
    try:
        thu["close"][0] = 1.0
    except pd.errors.ChainedAssignmentError as e:
        print(f"Bắt được: {type(e).__name__}")
        print(f"  {str(e)[:150]}")

Bắt được: ChainedAssignmentError
  A value is being set on a copy of a DataFrame or Series through chained assignment.
Such chained assignment never works to update the original DataFra


⚠️ Cách này **chỉ bắt được tình huống 1**. Tình huống 2 — ghi vào kết quả lọc —
không phát ra cảnh báo nào nên không có gì để biến thành lỗi. Với nó thì chỉ có
test kiểm chứng kết quả mới bắt được, và đó là lý do bạn cần test khẳng định
**giá trị**, không chỉ khẳng định "chạy không lỗi".

## Tổng kết

| Bạn cần | Viết |
|---|---|
| Ghi một ô | `df.loc[nhãn, "cột"] = x` |
| Ghi theo điều kiện | `df.loc[mask, "cột"] = x` |
| Tách ra sửa riêng | `sub = df[mask].copy()` |
| Thêm cột, giữ gốc | `df.assign(cot_moi=...)` |
| Lọc nhiều điều kiện | `df.query("a > 1 and b < 2")` |

**Năm điều mang sang notebook sau:**

1. **`.loc[0:2]` cho 3 dòng, `.iloc[0:2]` cho 2 dòng.** Lệch đúng một dòng —
   loại lỗi không ai nhìn thấy.
2. Boolean mask dùng `&` `|` `~`, và **luôn bọc ngoặc** từng điều kiện.
3. **Chained assignment có cảnh báo và không ghi.** Sửa bằng một lệnh `.loc`.
4. **Ghi vào kết quả lọc thì KHÔNG có cảnh báo** và cũng không ghi vào gốc.
   Đây là tình huống nguy hiểm nhất trong cả pandas 3.0.
5. Hàm biến đổi dữ liệu nên **trả về frame mới**, đừng sửa tại chỗ.

---

**Tiếp theo:** [`53_bien_doi_cot.ipynb`](53_bien_doi_cot.ipynb) — tạo và biến
đổi cột, và vì sao `apply(axis=1)` chậm hơn 500 lần.